# Zuo et al. (2020) CIV Batch PyQSOFit Workflow

This notebook reads `Target_lists/zuo_targets_data.csv`, validates the 19 Zuo et al. spectra, and retains 1445--1705 Angstrom for continuum fitting while fitting the CIV emission line over 1500--1700 Angstrom using model configurations A, B, and C from the primary sample (model D is excluded). The continuum windows are specified in `conti_windows`. Pixel rejection and absorption-rejection settings are specified in `q.Fit`; wavelength masking is disabled. It writes all selected-model results, standard PyQSOFit FITS outputs, and a one-row-per-target table selected by the minimum BIC.

> Note: a 1445--1705 Angstrom fit cannot independently measure the 1350 Angstrom continuum. The published `log_l1350_erg_s` and `log_l1350_err` values from the Zuo input table are therefore retained in the outputs as `zuo_log_l1350_erg_s` and `zuo_log_l1350_err`. Existing saved cell outputs may reflect the previous configuration; restart the kernel and rerun to apply these settings.

In [8]:
%matplotlib inline
from pathlib import Path
import tempfile
import timeit
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.io import fits
from astropy.table import Table

import sys
PROJECT_ROOT = Path('/Users/nayera/PyQSOFit')
sys.path.insert(0, str(PROJECT_ROOT))
from pyqsofit.PyQSOFit import QSOFit
from data_calculations import blueshift_calc, asym_calc

warnings.filterwarnings('ignore')
QSOFit.set_mpl_style()

In [9]:
# Input/output configuration
TARGET_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_targets_data.csv'
SPECTRA_ROOT = PROJECT_ROOT / 'CIV_galaxy_spectra_targets' / 'zuo_civ_data'
FIT_RESULTS_DIR = SPECTRA_ROOT / 'output'
TRACKING_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_batch_fit_inputs_1500_1700.csv'
RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_all_model_outputs_1500_1700.csv'
BEST_RESULTS_CSV = PROJECT_ROOT / 'Target_lists' / 'zuo_civ_best_bic_outputs_1500_1700.csv'
# Retain both continuum windows; CIV line priors still span 1500--1700 A.
WAVE_RANGE = np.array([1445.0, 1705.0])
CIV_LINE_RANGE = np.array([1500.0, 1700.0])
WAVE_MASK = None

# MCMC is intentionally run only once per target/model fit.
RUN_MCMC = True
NSAMP = 200
NBURN = 20
NTHIN = 10

# Match the primary-sample line-width constraints.
MAX_CIV_FWHM_KMS_SIGMA = 0.02157
NARROW_FWHM_LIMIT_KMS = 1600
NARROW_CIV_MAXSIG = 0.0022664260627611094
BROAD_FWHM_THRESHOLD = NARROW_FWHM_LIMIT_KMS

MODEL_GRID = {
    'A': {'br_component': 3, 'na_component': 0},
    'B': {'br_component': 2, 'na_component': 1},
    'C': {'br_component': 2, 'na_component': 0},
}

# Set to a small integer for a smoke test, then back to None for the full run.
TARGET_LIMIT = None


In [10]:
def as_serializable(value):
    if value is None:
        return ''
    arr = np.asarray(value)
    return arr.tolist()


REQUIRED_COLUMNS = {'name_sdss', 'z', 'ra', 'dec', 'plate', 'mjd', 'fiber', 'fits_path'}
SOURCE_COLUMNS = None


def inspect_spectrum(fits_path, z):
    if not fits_path.exists():
        return False, np.nan, np.nan, 'missing FITS file'
    try:
        with fits.open(fits_path) as hdul:
            names = set(hdul[1].columns.names)
            missing_arrays = sorted({'loglam', 'flux', 'ivar'} - names)
            if missing_arrays:
                return False, np.nan, np.nan, f'missing FITS arrays: {missing_arrays}'
            obs_wave = 10 ** np.asarray(hdul[1].data['loglam'], dtype=float)
        rest_min = float(np.nanmin(obs_wave) / (1 + z))
        rest_max = float(np.nanmax(obs_wave) / (1 + z))
        if rest_min > WAVE_RANGE[0] or rest_max < WAVE_RANGE[1]:
            reason = f'insufficient rest-frame coverage: {rest_min:.1f}--{rest_max:.1f} A'
            return False, rest_min, rest_max, reason
        return True, rest_min, rest_max, ''
    except Exception as exc:
        return False, np.nan, np.nan, f'cannot read FITS file: {exc!r}'


def load_targets(target_csv=TARGET_CSV):
    global SOURCE_COLUMNS
    df = pd.read_csv(target_csv)
    SOURCE_COLUMNS = list(df.columns)
    missing_columns = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing_columns:
        raise ValueError(f'Missing required columns in {target_csv.name}: {missing_columns}')

    df = df[df['name_sdss'].notna() & df['fits_path'].notna()].copy()
    df['target_name'] = df['name_sdss'].astype(str).str.strip()
    if df['target_name'].duplicated().any():
        duplicates = df.loc[df['target_name'].duplicated(False), 'target_name'].tolist()
        raise ValueError(f'Duplicate target names in {target_csv.name}: {duplicates}')
    df['file_name'] = df['fits_path'].apply(lambda value: Path(str(value)).name)
    # Resolve against the project folder so the notebook does not depend on CSV absolute paths.
    df['input_fits_path'] = df['fits_path'].astype(str)
    df['fits_path'] = df['file_name'].apply(lambda value: SPECTRA_ROOT / value)
    df['fit_z'] = pd.to_numeric(df['z'], errors='coerce')
    for column in ['ra', 'dec', 'plate', 'mjd', 'fiber']:
        df[column] = pd.to_numeric(df[column], errors='coerce')
    df['fit_z_source'] = 'Zuo2020 z'
    df['range_arr'] = [WAVE_RANGE.copy() for _ in range(len(df))]
    df['mask_arr'] = [None for _ in range(len(df))]

    inspections = []
    for row in df.itertuples(index=False):
        reasons = []
        if not np.isfinite(row.fit_z) or row.fit_z <= 0:
            valid, rest_min, rest_max = False, np.nan, np.nan
            reasons.append('missing/non-positive fit redshift')
        else:
            valid, rest_min, rest_max, spectrum_reason = inspect_spectrum(
                Path(row.fits_path), float(row.fit_z)
            )
            if spectrum_reason:
                reasons.append(spectrum_reason)
        missing_metadata = [
            column for column in ['ra', 'dec', 'plate', 'mjd', 'fiber']
            if not np.isfinite(getattr(row, column))
        ]
        if missing_metadata:
            valid = False
            reasons.append(f'missing/non-numeric metadata: {missing_metadata}')
        inspections.append((valid, rest_min, rest_max, '; '.join(reasons)))
    df[['spectrum_valid', 'rest_wave_min', 'rest_wave_max', 'skip_reason']] = pd.DataFrame(
        inspections, index=df.index
    )
    df['run_fit'] = df['spectrum_valid'] & df['fit_z'].notna() & (df['fit_z'] > 0)

    export = df.copy()
    export['fits_path'] = export['fits_path'].astype(str)
    export['range_arr'] = export['range_arr'].apply(as_serializable)
    export['mask_arr'] = export['mask_arr'].apply(as_serializable)
    export.to_csv(TRACKING_CSV, index=False)
    return df


targets = load_targets()
fit_targets = targets[targets['run_fit']].copy()
if TARGET_LIMIT is not None:
    fit_targets = fit_targets.head(TARGET_LIMIT).copy()

print(f'Loaded {len(targets)} Zuo et al. targets from {TARGET_CSV.name}.')
print(f'{len(fit_targets)} targets passed the FITS/redshift/coverage preflight.')
print(f'Tracking table written to {TRACKING_CSV}.')
if (~targets['run_fit']).any():
    display(targets.loc[~targets['run_fit'], ['target_name', 'file_name', 'fit_z', 'skip_reason']])
    raise RuntimeError('Preflight failed; fix the listed inputs before starting the batch fits.')

display(fit_targets[['target_name', 'file_name', 'fit_z', 'range_arr', 'mask_arr', 'rest_wave_min', 'rest_wave_max', 'fits_path']])

Loaded 19 Zuo et al. targets from zuo_targets_data.csv.
19 targets passed the FITS/redshift/coverage preflight.
Tracking table written to /Users/nayera/PyQSOFit/Target_lists/zuo_civ_batch_fit_inputs_1500_1700.csv.


,target_name,file_name,fit_z,range_arr,mask_arr,rest_wave_min,rest_wave_max,fits_path
0,J011521.20+152453.3,spec-2825-54439-0077.fits,3.443,"[1445.0, 1705.0]",None,857.282075,2069.301200,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
1,J014214.75+002324.2,spec-0401-51788-0500.fits,3.379,"[1445.0, 1705.0]",None,865.416128,2099.544470,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
2,J015741.57-010629.6,spec-4233-55449-0178.fits,3.572,"[1445.0, 1705.0]",None,779.818837,2257.843950,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
3,J025021.76-075749.9,spec-0457-51901-0261.fits,3.337,"[1445.0, 1705.0]",None,879.043693,2118.900520,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
4,J025905.63+001121.9,spec-4242-55476-0977.fits,3.373,"[1445.0, 1705.0]",None,821.713748,2374.218715,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
5,J030341.04-002321.9,spec-0709-52205-0106.fits,3.233,"[1445.0, 1705.0]",None,897.949179,2173.460166,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
6,J030449.85-000813.4,spec-0411-51817-0153.fits,3.287,"[1445.0, 1705.0]",None,887.659707,2144.601174,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
7,J075303.34+423130.8,spec-3683-55178-0484.fits,3.590,"[1445.0, 1705.0]",None,775.867024,2248.989660,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
8,J080430.56+542041.1,spec-1871-53384-0234.fits,3.759,"[1445.0, 1705.0]",None,797.782229,1929.230941,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...
9,J080819.69+373047.3,spec-3804-55267-0640.fits,3.480,"[1445.0, 1705.0]",None,799.322222,2310.584483,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...


In [11]:
def continuum_windows_from_range(range_arr):
    lo, hi = map(float, range_arr)
    low_start = max(lo, 1445.0)
    low_end = min(low_start + 20.0, hi)
    high_end = hi
    high_start = max(hi - 15.0, low_end + 5.0)
    if low_end - low_start < 5.0 or high_end - high_start < 5.0 or high_start <= low_end:
        width = hi - lo
        low_start, low_end = lo, lo + 0.15 * width
        high_start, high_end = hi - 0.15 * width, hi
    return np.rec.array(
        [(low_start, low_end), (high_start, high_end)],
        formats='float32,float32',
        names='min,max',
    )

conti_windows = np.rec.array([    
    # CIV continuum fitting windows
    (1445., 1465.), #1445, 1465 | 1455, 1465
    (1700., 1705.), #1700, 1705 | 1585., 1600.

                            # Continuum fitting windows (to avoid emission line, etc.)  [AA]
    ], 
    formats = 'float32,  float32',
    names =    'min,     max')


def make_qsopar(param_dir, range_arr, br_component, na_component):
    primary_hdu = fits.PrimaryHDU(header=fits.Header({'Author': 'Nayera Abdessalam'}))

    line_priors = np.rec.array([
        (1549.06, 'CIV', 1500, 1700, 'CIV_br', br_component, 0.0, 0.0, 1e10,
         0.005, 0.004, MAX_CIV_FWHM_KMS_SIGMA, 0.015, 0, 0, 0, 0.05, 1),
        (1549.06, 'CIV', 1500, 1700, 'CIV_na', na_component, 0.0, 0.0, 1e10,
         1e-3, 5e-4, NARROW_CIV_MAXSIG, 0.01, 1, 1, 0, 0.002, 1),
    ], formats='float32,a20,float32,float32,a20,int32,float32,float32,float32,float32,float32,float32,float32,int32,int32,int32,float32,int32',
       names='lambda,compname,minwav,maxwav,linename,ngauss,inisca,minsca,maxsca,inisig,minsig,maxsig,voff,vindex,windex,findex,fvalue,vary')
    hdu1 = fits.BinTableHDU(data=line_priors, name='line_priors')

    hdu2 = fits.BinTableHDU(data=conti_windows, name='conti_windows')

    conti_priors = np.rec.array([
        ('Fe_uv_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_uv_FWHM', 3000, 1200, 18000, 1),
        ('Fe_uv_shift', 0.0, -0.01, 0.01, 1),
        ('Fe_op_norm', 0.0, 0.0, 1e10, 1),
        ('Fe_op_FWHM', 3000, 1200, 18000, 1),
        ('Fe_op_shift', 0.0, -0.01, 0.01, 1),
        ('PL_norm', 1.0, 0.0, 1e10, 1),
        ('PL_slope', -1.5, -5.0, 3.0, 1),
        ('Blamer_norm', 0.0, 0.0, 1e10, 1),
        ('Balmer_Te', 15000, 10000, 50000, 1),
        ('Balmer_Tau', 0.5, 0.1, 2.0, 1),
        ('conti_a_0', 0.0, None, None, 1),
        ('conti_a_1', 0.0, None, None, 1),
        ('conti_a_2', 0.0, None, None, 1),
    ], formats='a20,float32,float32,float32,int32', names='parname,initial,min,max,vary')
    hdu3 = fits.BinTableHDU(data=conti_priors, name='conti_priors')

    measure_info = Table([[[1350, 1450, 3000, 4200, 5100]], [[[1500, 1550]]]],
                         names=['cont_loc', 'Fe_flux_range'], dtype=['float32', 'float32'])
    hdu4 = fits.BinTableHDU(data=measure_info, name='measure_info')

    param_path = Path(param_dir) / 'qsopar.fits'
    fits.HDUList([primary_hdu, hdu1, hdu2, hdu3, hdu4]).writeto(param_path, overwrite=True)
    return param_path


In [12]:
def read_sdss_spectrum(fits_path):
    with fits.open(fits_path) as data:
        lam = 10 ** data[1].data['loglam']
        flux = np.asarray(data[1].data['flux'], dtype=float)
        ivar = np.asarray(data[1].data['ivar'], dtype=float)
        err = np.full_like(flux, np.inf, dtype=float)
        good_ivar = np.isfinite(ivar) & (ivar > 0)
        err[good_ivar] = 1 / np.sqrt(ivar[good_ivar])


    # No S/N cut or down-weighting: preserve every pixel with a usable inverse variance.
    usable = np.isfinite(flux) & np.isfinite(err) & (err > 0) & (flux != 0)
    meta = {
        'usable_pixels': int(np.sum(usable)),
        'unusable_pixels': int(np.sum(~usable)),
        'total_pixels': int(len(flux)),
    }
    return lam, flux, err, meta


def named_pyqsofit_results(q):
    """Return every scalar continuum and line result with collision-safe names."""
    output = {}
    for prefix, names, values in (
        ('pyqsofit_conti', q.conti_result_name, q.conti_result),
        ('pyqsofit_line', q.line_result_name, q.line_result),
    ):
        for name, value in zip(names, values):
            scalar = np.ravel(value)[0]
            output[f'{prefix}_{name}'] = scalar.item() if hasattr(scalar, 'item') else scalar
    return output


def source_metadata(target_row):
    """Carry the complete Zuo input row into each result for combined analysis."""
    return {
        f'zuo_{column}': (target_row['input_fits_path'] if column == 'fits_path' else target_row[column])
        for column in SOURCE_COLUMNS
    }


def get_line_value(q, key, default=np.nan):
    idx = np.where(np.asarray(q.line_result_name) == key)[0]
    if idx.size == 0:
        return default
    return float(np.ravel(q.line_result[idx[0]])[0])


def component_props(q, line_name, line_type, ln_sigma_br=NARROW_CIV_MAXSIG):
    names = ['FWHM', 'sigma', 'EW', 'peak', 'area', 'snr']
    values = np.array(q.line_prop_from_name(line_name, line_type, ln_sigma_br=ln_sigma_br), dtype=float)
    out = dict(zip(names, values))

    if getattr(q, 'MCMC', False) and len(getattr(q, 'gauss_result_all', [])) > 0:
        samples = []
        for i in range(q.gauss_result_all.shape[0]):
            samples.append(q.line_prop_from_name(line_name, line_type, sample_index=i, ln_sigma_br=ln_sigma_br))
        samples = np.asarray(samples, dtype=float)
        for j, name in enumerate(names):
            finite = samples[:, j][np.isfinite(samples[:, j])]
            if finite.size:
                lo, hi = np.nanpercentile(finite, [16, 84])
                out[f'{name}_err_mcmc'] = float((hi - lo) / 2)
            else:
                out[f'{name}_err_mcmc'] = np.nan
    else:
        for name in names:
            out[f'{name}_err_mcmc'] = np.nan
    return out


def gaussian_params(q, line_name, sample_index=None):
    result_names = np.asarray(q.gauss_result_name)
    result = np.asarray(q.gauss_result, dtype=object)
    params = []
    i = 1
    while True:
        keys = [f'{line_name}_{i}_scale', f'{line_name}_{i}_centerwave', f'{line_name}_{i}_sigma']
        if not all(key in result_names for key in keys):
            break
        if sample_index is None:
            vals = [float(np.ravel(result[result_names == key][0])[0]) for key in keys]
        else:
            # gauss_result_name interleaves value/error, while gauss_result_all stores values only.
            indices = [int(np.flatnonzero(result_names == key)[0] // 2) for key in keys]
            vals = [float(q.gauss_result_all[sample_index, index]) for index in indices]
        params.append(vals)
        i += 1
    return np.asarray(params, dtype=float)


def broad_half_wavelengths(q, sample_index=None):
    params = gaussian_params(q, 'CIV_br', sample_index=sample_index)
    if params.size == 0:
        return np.nan, np.nan, np.nan
    lam = np.asarray(q.wave, dtype=float)
    sel = (lam >= CIV_LINE_RANGE[0]) & (lam <= CIV_LINE_RANGE[1])
    if np.sum(sel) < 3:
        sel = np.ones_like(lam, dtype=bool)
    lam_w = lam[sel]
    lnlam = np.log(lam_w)
    model = np.zeros_like(lam_w, dtype=float)
    for amp, center, sigma in params:
        model += amp * np.exp(-0.5 * ((lnlam - center) / sigma) ** 2)
    if not np.any(np.isfinite(model)) or np.nanmax(model) <= 0:
        return np.nan, np.nan, np.nan

    peak = np.nanmax(model)
    i_peak = int(np.nanargmax(model))
    half_peak = 0.5 * peak

    def crossing_left():
        below = np.where(model[:i_peak] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[0])
        j0 = below[-1]
        j1 = j0 + 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    def crossing_right():
        below = np.where(model[i_peak + 1:] < half_peak)[0]
        if below.size == 0:
            return float(lam_w[-1])
        j1 = i_peak + 1 + below[0]
        j0 = j1 - 1
        return float(np.interp(half_peak, [model[j0], model[j1]], [lam_w[j0], lam_w[j1]]))

    lam_blue = crossing_left()
    lam_red = crossing_right()

    increments = 0.5 * (model[1:] + model[:-1]) * np.diff(lam_w)
    cum = np.cumsum(increments)
    if cum.size == 0 or cum[-1] <= 0:
        lam_half = np.nan
    else:
        lam_half = float(np.interp(0.5 * cum[-1], cum, lam_w[1:]))
    return lam_blue, lam_red, lam_half


def posterior_error(values):
    finite = np.asarray(values, dtype=float)
    finite = finite[np.isfinite(finite)]
    if finite.size == 0:
        return np.nan
    lo, hi = np.nanpercentile(finite, [16, 84])
    return float((hi - lo) / 2)


def broad_profile_mcmc_results(q):
    samples = getattr(q, 'gauss_result_all', None)
    if not getattr(q, 'MCMC', False) or samples is None or len(samples) == 0:
        return {
            'CIV_blueshift_err_mcmc': np.nan,
            'CIV_asym_err_mcmc': np.nan,
            'lam_blue_err_mcmc': np.nan,
            'lam_red_err_mcmc': np.nan,
            'lam_half_err_mcmc': np.nan,
            'mcmc_samples_returned': 0,
            'mcmc_valid_blueshift_samples': 0,
            'mcmc_valid_asym_samples': 0,
        }

    n_samples = samples.shape[0]
    profiles = [broad_half_wavelengths(q, sample_index=i) for i in range(n_samples)]
    lam_blue = np.asarray([profile[0] for profile in profiles], dtype=float)
    lam_red = np.asarray([profile[1] for profile in profiles], dtype=float)
    lam_half = np.asarray([profile[2] for profile in profiles], dtype=float)

    # Use PyQSOFit's continuous-model broad-line peak for each posterior sample.
    broad_properties = np.asarray([
        q.line_prop_from_name(
            'CIV_br', 'broad', sample_index=i, ln_sigma_br=NARROW_CIV_MAXSIG
        )
        for i in range(n_samples)
    ], dtype=float)
    br_peak = broad_properties[:, 3]

    blueshift = np.asarray([blueshift_calc(value) for value in lam_half], dtype=float)
    # asym_calc is imported from data_calculations.py, the authoritative equation.
    asymmetry = np.asarray([
        asym_calc(blue, red, peak)
        for blue, red, peak in zip(lam_blue, lam_red, br_peak)
    ], dtype=float)

    return {
        'CIV_blueshift_err_mcmc': posterior_error(blueshift),
        'CIV_asym_err_mcmc': posterior_error(asymmetry),
        'lam_blue_err_mcmc': posterior_error(lam_blue),
        'lam_red_err_mcmc': posterior_error(lam_red),
        'lam_half_err_mcmc': posterior_error(lam_half),
        'mcmc_samples_returned': int(n_samples),
        'mcmc_valid_blueshift_samples': int(np.isfinite(blueshift).sum()),
        'mcmc_valid_asym_samples': int(np.isfinite(asymmetry).sum()),
    }


In [13]:
def fit_target_model(target_row, model_name, model_cfg):
    target = target_row['target_name']
    fits_path = Path(target_row['fits_path'])
    z = float(target_row['fit_z'])
    range_arr = np.asarray(target_row['range_arr'], dtype=float)
    mask_arr = WAVE_MASK
    output_stem = f"{model_name}_{fits_path.stem}"
    output_fits_path = FIT_RESULTS_DIR / f'{output_stem}.fits'

    lam, flux, err, meta = read_sdss_spectrum(fits_path)

    with tempfile.TemporaryDirectory(prefix='pyqsofit_civ_') as param_dir:
        make_qsopar(param_dir, range_arr, model_cfg['br_component'], model_cfg['na_component'])
        q = QSOFit(lam, flux, err, z,
                   ra=float(target_row['ra']), dec=float(target_row['dec']),
                   plateid=int(target_row['plate']), mjd=int(target_row['mjd']),
                   fiberid=int(target_row['fiber']), path=param_dir)

        start = timeit.default_timer()
        q.Fit(name=f'{target}_{model_name}',
              nsmooth=1,
              and_mask=False,
              or_mask=False,
              reject_badpix=True,  # GESD outliers, not 5-sigma/20-pixel absorption masking
              deredden=False,
              wave_range=range_arr,
              wave_mask=mask_arr,
              decompose_host=False,
              host_prior=False,
              host_prior_scale=0.2,
              host_line_mask=True,
              decomp_na_mask=True,
              qso_type='CZBIN1',
              npca_qso=10,
              host_type='PCA',
              npca_gal=5,
              Fe_uv_op=True,
              poly=False,
              BC=False,
              initial_guess=None,
              rej_abs_conti=False,
              n_pix_min_conti=30,  # Allow UV FeII fitting with these narrow continuum windows; normalization may remain zero
              linefit=True,
              rej_abs_line=True,
              MC=False,
              MCMC=RUN_MCMC,
              nsamp=NSAMP,
              param_file_name='qsopar.fits',
              nburn=NBURN,
              nthin=NTHIN,
              epsilon_jitter=0.,
              save_result=True,
              save_fits_path=str(FIT_RESULTS_DIR),
              save_fits_name=output_stem,
              plot_fig=True,
              save_fig=True,
              plot_corner=False,
              verbose=False,
              kwargs_plot={'broad_fwhm': BROAD_FWHM_THRESHOLD, 
                           'save_fig_path': str(FIT_RESULTS_DIR),},
              kwargs_conti_emcee={},
              kwargs_line_emcee={})
        fit_seconds = float(timeit.default_timer() - start)

    br = component_props(q, 'CIV_br', 'broad', ln_sigma_br=NARROW_CIV_MAXSIG)
    na = component_props(q, 'CIV_na', 'narrow', ln_sigma_br=NARROW_CIV_MAXSIG)
    lam_blue, lam_red, lam_half = broad_half_wavelengths(q)
    blueshift = blueshift_calc(lam_half)
    asymmetry = asym_calc(lam_blue, lam_red, br['peak'])
    profile_mcmc = broad_profile_mcmc_results(q)

    row = {
        'Target': target,
        'Model': model_name,
        'file_name': target_row['file_name'],
        'fits_path': str(fits_path),
        'fit_z': z,
        'fit_z_source': target_row['fit_z_source'],
        'range_arr': as_serializable(range_arr),
        'mask_arr': as_serializable(mask_arr),
        'output_fits_path': str(output_fits_path),
        'br_components_forced': model_cfg['br_component'],
        'na_components_forced': model_cfg['na_component'],
        'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
        'lam_blue': lam_blue,
        'lam_red': lam_red,
        'lam_half': lam_half,
        'br_peak': br['peak'],
        'br_peak_err_mcmc': br['peak_err_mcmc'],
        'CIV_blueshift_kms': blueshift,
        'CIV_blueshift_err_mcmc': profile_mcmc['CIV_blueshift_err_mcmc'],
        'CIV_asym': asymmetry,
        'CIV_asym_err_mcmc': profile_mcmc['CIV_asym_err_mcmc'],
        'lam_blue_err_mcmc': profile_mcmc['lam_blue_err_mcmc'],
        'lam_red_err_mcmc': profile_mcmc['lam_red_err_mcmc'],
        'lam_half_err_mcmc': profile_mcmc['lam_half_err_mcmc'],
        'mcmc_samples_returned': profile_mcmc['mcmc_samples_returned'],
        'mcmc_valid_blueshift_samples': profile_mcmc['mcmc_valid_blueshift_samples'],
        'mcmc_valid_asym_samples': profile_mcmc['mcmc_valid_asym_samples'],
        'reduced Chi squared': get_line_value(q, '1_line_red_chi2'),
        'BIC': get_line_value(q, '1_line_bic'),
        'BIC_mcmc': get_line_value(q, '1_line_bic'),
        'FWHM_CIV': br['FWHM'],
        'FWHM_err_mcmc': br['FWHM_err_mcmc'],
        'br_EW': br['EW'],
        'br_EW_err_mcmc': br['EW_err_mcmc'],
        'br_area': br['area'],
        'br_area_err_mcmc': br['area_err_mcmc'],
        'br_sigma': br['sigma'],
        'br_sigma_err_mcmc': br['sigma_err_mcmc'],
        'br_snr': br['snr'],
        'na_FWHM': na['FWHM'],
        'na_FWHM_err_mcmc': na['FWHM_err_mcmc'],
        'na_EW': na['EW'],
        'na_EW_err_mcmc': na['EW_err_mcmc'],
        'na_area': na['area'],
        'na_area_err_mcmc': na['area_err_mcmc'],
        'na_sigma': na['sigma'],
        'na_sigma_err_mcmc': na['sigma_err_mcmc'],
        'na_peak': na['peak'],
        'na_peak_err_mcmc': na['peak_err_mcmc'],
        'na_snr': na['snr'],
        'line_min_chi2': get_line_value(q, '1_line_min_chi2'),
        'line_status': get_line_value(q, '1_line_status'),
        'line_ndof': get_line_value(q, '1_ndof'),
        'line_niter': get_line_value(q, '1_niter'),
        'usable_pixels': meta['usable_pixels'],
        'unusable_pixels': meta['unusable_pixels'],
        'total_pixels': meta['total_pixels'],
        'fit_seconds': fit_seconds,
        'status': 'ok',
        'error': '',
    }
    row.update(source_metadata(target_row))
    row.update(named_pyqsofit_results(q))
    return row


In [14]:
FIT_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results = []
expected = len(fit_targets) * len(MODEL_GRID)
print(f'Running {expected} fits: {len(fit_targets)} targets x {len(MODEL_GRID)} models.')

for _, target_row in fit_targets.iterrows():
    for model_name, model_cfg in MODEL_GRID.items():
        label = f"{target_row['target_name']} model {model_name}"
        print(f'Starting {label}')
        try:
            row = fit_target_model(target_row, model_name, model_cfg)
            print(
                f"Finished {label}: BIC={row['BIC']:.3g}, "
                f"FWHM={row['FWHM_CIV']:.3g} km/s, "
                f"blueshift_err={row['CIV_blueshift_err_mcmc']:.3g} km/s, "
                f"asym_err={row['CIV_asym_err_mcmc']:.3g}"
            )
        except Exception as exc:
            row = {
                'Target': target_row['target_name'],
                'Model': model_name,
                'file_name': target_row['file_name'],
                'fits_path': str(target_row['fits_path']),
                'fit_z': target_row['fit_z'],
                'fit_z_source': target_row['fit_z_source'],
                'range_arr': as_serializable(target_row['range_arr']),
                'mask_arr': as_serializable(target_row['mask_arr']),
                'output_fits_path': str(FIT_RESULTS_DIR / f"{model_name}_{Path(target_row['file_name']).stem}.fits"),
                'br_components_forced': model_cfg['br_component'],
                'na_components_forced': model_cfg['na_component'],
                'narrow_fwhm_limit_kms': NARROW_FWHM_LIMIT_KMS,
                'status': 'failed',
                'error': repr(exc),
            }
            row.update(source_metadata(target_row))
            print(f'FAILED {label}: {exc!r}')
        results.append(row)
        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)

results_df = pd.DataFrame(results)
if 'BIC' not in results_df:
    results_df['BIC'] = np.nan
else:
    results_df['BIC'] = pd.to_numeric(results_df['BIC'], errors='coerce')
successful = results_df['status'].eq('ok') & np.isfinite(results_df['BIC'])
results_df['delta_BIC'] = np.nan
results_df['is_best_BIC'] = False
if successful.any():
    best_bic_by_target = results_df.loc[successful].groupby('Target')['BIC'].transform('min')
    results_df.loc[successful, 'delta_BIC'] = (
        results_df.loc[successful, 'BIC'] - best_bic_by_target
    )
    best_indices = results_df.loc[successful].groupby('Target', sort=False)['BIC'].idxmin()
    results_df.loc[best_indices, 'is_best_BIC'] = True
    best_results_df = results_df.loc[best_indices].copy().reset_index(drop=True)
else:
    best_results_df = results_df.iloc[0:0].copy()

results_df.to_csv(RESULTS_CSV, index=False)
best_results_df.to_csv(BEST_RESULTS_CSV, index=False)
missing_best = sorted(set(fit_targets['target_name']) - set(best_results_df['Target']))
print(f'Wrote {len(results_df)} selected-model rows to {RESULTS_CSV}')
print(f'Wrote {len(best_results_df)} minimum-BIC rows to {BEST_RESULTS_CSV}')
if missing_best:
    print(f'WARNING: no successful finite-BIC model for: {missing_best}')
display(best_results_df)


Running 57 fits: 19 targets x 3 models.
Starting J011521.20+152453.3 model A


100%|██████████| 200/200 [00:13<00:00, 14.90it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 5 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [20.98611019 25.1451486  21.4138753  22.38633589 23.98094078]


100%|██████████| 200/200 [00:01<00:00, 126.61it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 9 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [19.01489233 20.42996928 19.87829997 18.50724096 18.4221454  19.25719881
 18.19841975 17.99177812 18.80112571]
Finished J011521.20+152453.3 model A: BIC=153, FWHM=7.43e+03 km/s, blueshift_err=29.7 km/s, asym_err=0.0572
Starting J011521.20+152453.3 model B


100%|██████████| 200/200 [00:13<00:00, 15.03it/s]


... (220 outputs left out)

The chain is shorter than 50 times the integrated autocorrelation time for 5 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [15.93072157 18.73150961 26.91177347 26.57690842 25.09114495]


100%|██████████| 200/200 [00:01<00:00, 118.08it/s]


The chain is shorter than 50 times the integrated autocorrelation time for 6 parameter(s). Use this estimate with caution and run a longer chain!
N/50 = 4;
tau: [16.97706614 17.83656058 19.79040854 17.17509271 16.12166788 16.29456038]
Finished J230301.45-093930.7 model C: BIC=169, FWHM=9.46e+03 km/s, blueshift_err=54.2 km/s, asym_err=0.0618
Wrote 57 selected-model rows to /Users/nayera/PyQSOFit/Target_lists/zuo_civ_all_model_outputs_1500_1700.csv
Wrote 19 minimum-BIC rows to /Users/nayera/PyQSOFit/Target_lists/zuo_civ_best_bic_outputs_1500_1700.csv


,Target,Model,file_name,fits_path,fit_z,fit_z_source,range_arr,mask_arr,output_fits_path,br_components_forced,...,pyqsofit_line_CIV_whole_br_snr,pyqsofit_line_CIV_whole_br_snr_err,pyqsofit_line_CIV_na_1_scale,pyqsofit_line_CIV_na_1_scale_err,pyqsofit_line_CIV_na_1_centerwave,pyqsofit_line_CIV_na_1_centerwave_err,pyqsofit_line_CIV_na_1_sigma,pyqsofit_line_CIV_na_1_sigma_err,delta_BIC,is_best_BIC
0,J011521.20+152453.3,A,spec-2825-54439-0077.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.443,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3,...,14.130542364637117,0.2585145717237145,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
1,J014214.75+002324.2,A,spec-0401-51788-0500.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.379,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3,...,8.805137867716601,0.17697999215047844,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
2,J015741.57-010629.6,A,spec-4233-55449-0178.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.572,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3,...,19.179668381276148,0.2514372715828035,NaN,NaN,NaN,NaN,NaN,NaN,0.0,True
3,J025021.76-075749.9,B,spec-0457-51901-0261.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.337,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,11.159106546581631,0.20610391792023997,21.57809209091255,0.6541617976047469,7.345959560623134,2.4383628296931192e-05,0.0022664260468175405,3.269945767215058e-05,0.0,True
4,J025905.63+001121.9,B,spec-4242-55476-0977.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.373,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,57.23977736525438,0.680397052250246,78.02929080646948,1.4391055415311342,7.3453684877876855,5.1863091954373886e-06,0.0021938442479037335,4.21791987001356e-05,0.0,True
5,J030341.04-002321.9,B,spec-0709-52205-0106.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.233,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,13.925195707575455,0.16625009823527037,14.087097044424013,0.28890447594947677,7.343213679685248,6.262321142180838e-05,0.002266426046835233,1.810881775141885e-05,0.0,True
6,J030449.85-000813.4,B,spec-0411-51817-0153.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.287,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,28.426604631534236,0.43896282256423014,94.23570646038826,2.6130435565855805,7.345029832395109,2.27518305306873e-05,0.0022664250757293896,2.288727927485701e-05,0.0,True
7,J075303.34+423130.8,B,spec-3683-55178-0484.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.590,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,29.42004867277382,0.3892320421687536,47.693731186626565,0.7531814451991394,7.346011931877943,1.2825386127790495e-05,0.0022664260467349048,8.92811422045954e-06,0.0,True
8,J080430.56+542041.1,B,spec-1871-53384-0234.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.759,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,6.644671434973428,0.12133634017481709,8.746342539112106,0.39361396458205977,7.3440848604744415,6.371788532932854e-05,0.002266426046719832,2.8207063664812716e-05,0.0,True
9,J080819.69+373047.3,B,spec-3804-55267-0640.fits,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,3.480,Zuo2020 z,"[1445.0, 1705.0]",,/Users/nayera/PyQSOFit/CIV_galaxy_spectra_targ...,2,...,8.100068364571431,0.23252050525325485,11.52747675092769,0.3791778279293512,7.338553919493654,0.00014588358924960687,0.002266426045623514,3.4666424515775434e-05,0.0,True


## Outputs

`zuo_civ_all_model_outputs_1500_1700.csv` contains all four candidate fits per target. `zuo_civ_best_bic_outputs_1500_1700.csv` contains the minimum-BIC fit for each target. Every original Zuo table field is retained with a `zuo_` prefix, and every scalar returned by PyQSOFit is retained with a `pyqsofit_conti_` or `pyqsofit_line_` prefix. Standard PyQSOFit result FITS files are written under `CIV_galaxy_spectra_targets/zuo_civ_data/output/`.